In [1]:
import pandas as pd
import numpy as np
import re
import random
from pathlib import Path

# Đảm bảo thư mục outputs tồn tại
Path('../outputs').mkdir(exist_ok=True)

print("🔄 Đang tải dữ liệu đã làm sạch từ Tuần 1...")

# Tải dữ liệu
df_jobs = pd.read_csv('../data/processed/jobs_tech_clean.csv')
df_resumes = pd.read_csv('../data/processed/resumes_clean.csv')

print(f"✅ Loaded {len(df_jobs)} Jobs và {len(df_resumes)} Resumes.")
print(f"📊 Cột trong Jobs: {df_jobs.columns.tolist()}")
print(f"📊 Cột trong Resumes: {df_resumes.columns.tolist()}")

🔄 Đang tải dữ liệu đã làm sạch từ Tuần 1...
✅ Loaded 29101 Jobs và 2483 Resumes.
📊 Cột trong Jobs: ['job_id', 'job_title', 'job_description', 'job_skills', 'location', 'clean_job_desc', 'clean_job_skills', 'word_count']
📊 Cột trong Resumes: ['resume_id', 'resume_text', 'category', 'clean_resume_text', 'word_count']


In [2]:
print("=" * 60)
print("BƯỚC 1: TRÍCH XUẤT KỸ NĂNG (SKILL EXTRACTION)")
print("=" * 60)

# Từ điển kỹ năng IT phổ biến (Lấy từ kết quả EDA Tuần 1 và kiến thức thực tế)
TECH_SKILLS = [
    'python', 'java', 'javascript', 'c++', 'c#', 'ruby', 'go', 'golang', 'rust', 'swift', 'kotlin', 'php',
    'sql', 'nosql', 'mongodb', 'postgresql', 'mysql', 'oracle',
    'machine learning', 'deep learning', 'nlp', 'computer vision', 'data analysis', 'data science', 'ai',
    'tensorflow', 'pytorch', 'scikit-learn', 'keras', 'pandas', 'numpy', 'matplotlib', 'seaborn',
    'aws', 'azure', 'gcp', 'cloud', 'docker', 'kubernetes', 'jenkins', 'ci/cd', 'devops', 'linux', 'bash',
    'react', 'angular', 'vue', 'node.js', 'express', 'django', 'flask', 'spring boot',
    'html', 'css', 'rest api', 'graphql', 'git', 'github', 'jira', 'agile', 'scrum',
    'tableau', 'power bi', 'excel', 'sap', 'salesforce',
    'cybersecurity', 'networking', 'tcp/ip', 'firewall'
]

# Hàm trích xuất kỹ năng từ văn bản
def extract_skills(text):
    if not isinstance(text, str):
        return []
    text_lower = text.lower()
    found_skills = []
    for skill in TECH_SKILLS:
        # Dùng regex để tìm chính xác từ khóa, tránh lỗi substring (ví dụ: 'c' trong 'cat')
        if re.search(r'\b' + re.escape(skill) + r'\b', text_lower):
            found_skills.append(skill)
    return list(set(found_skills)) # Loại bỏ trùng lặp

# Áp dụng cho Jobs và Resumes
print("🔍 Đang quét kỹ năng trong Job Descriptions...")
df_jobs['extracted_skills'] = df_jobs['clean_job_desc'].apply(extract_skills)

print(" Đang quét kỹ năng trong Resumes...")
df_resumes['extracted_skills'] = df_resumes['clean_resume_text'].apply(extract_skills)

# Tạo thêm cột số lượng kỹ năng tìm được
df_jobs['skill_count'] = df_jobs['extracted_skills'].apply(len)
df_resumes['skill_count'] = df_resumes['extracted_skills'].apply(len)

print(f"\n✅ Hoàn tất! Trung bình mỗi Job có {df_jobs['skill_count'].mean():.1f} kỹ năng được nhận diện.")
print(f"✅ Trung bình mỗi CV có {df_resumes['skill_count'].mean():.1f} kỹ năng được nhận diện.")

# Xem thử kết quả
print("\n--- MẪU KỸ NĂNG TRONG JOB ---")
display(df_jobs[['job_title', 'extracted_skills']].head(3))

BƯỚC 1: TRÍCH XUẤT KỸ NĂNG (SKILL EXTRACTION)
🔍 Đang quét kỹ năng trong Job Descriptions...
 Đang quét kỹ năng trong Resumes...

✅ Hoàn tất! Trung bình mỗi Job có 2.1 kỹ năng được nhận diện.
✅ Trung bình mỗi CV có 1.3 kỹ năng được nhận diện.

--- MẪU KỸ NĂNG TRONG JOB ---


,job_title,extracted_skills
0,Building Engineer,[]
1,Structural Engineer,[]
2,Software Engineer,[]


In [3]:
print("=" * 60)
print("BƯỚC 2: XÂY DỰNG GOLDEN TEST SET (GROUND TRUTH)")
print("=" * 60)

# 1. Định nghĩa từ khóa ánh xạ giữa Category của Resume và Title của Job
category_job_mapping = {
    'Information Technology': ['software', 'developer', 'engineer', 'it ', 'architect', 'devops', 'data'],
    'Data Science': ['data scientist', 'data analyst', 'machine learning', 'ai ', 'data engineer'],
    'HR': ['hr ', 'human resource', 'recruiter', 'talent'],
    'Finance': ['finance', 'accountant', 'financial', 'audit', 'tax'],
    'Marketing': ['marketing', 'seo', 'content', 'digital marketing', 'brand'],
    'Sales': ['sales', 'business development', 'account executive', 'bde'],
    'Engineering': ['mechanical', 'electrical', 'civil', 'engineering']
}

# Hàm kiểm tra xem Job title có khớp với Category của Resume không
def is_matching_pair(resume_category, job_title):
    job_title_lower = str(job_title).lower()
    
    # Tìm category khớp nhất
    for cat, keywords in category_job_mapping.items():
        if cat.lower() in resume_category.lower():
            # Nếu tìm thấy category, kiểm tra xem job title có chứa từ khóa của category đó không
            for keyword in keywords:
                if keyword in job_title_lower:
                    return 1 # Match (Positive pair)
            return 0 # Mismatch (Negative pair)
            
    return 0 # Không xác định được, coi như mismatch cho an toàn

# 2. Tạo các cặp (Pairs)
# Để đảm bảo công bằng, chúng ta sẽ tạo 50% cặp Match và 50% cặp Mismatch
pairs_data = []

print("️ Đang tạo cặp dữ liệu (Có thể mất 1-2 phút)...")

# Lấy mẫu ngẫu nhiên để tăng tốc độ xử lý (Ví dụ: 500 Jobs và 500 Resumes)
sample_jobs = df_jobs.sample(n=min(500, len(df_jobs)), random_state=42)
sample_resumes = df_resumes.sample(n=min(500, len(df_resumes)), random_state=42)

# Tạo Positive Pairs (Match)
for _, job in sample_jobs.iterrows():
    for _, resume in sample_resumes.iterrows():
        label = is_matching_pair(resume['category'], job['job_title'])
        if label == 1:
            pairs_data.append({
                'resume_text': resume['clean_resume_text'],
                'job_text': job['clean_job_desc'],
                'label': 1,
                'resume_category': resume['category'],
                'job_title': job['job_title']
            })

# Tạo Negative Pairs (Mismatch) - Lấy ngẫu nhiên các cặp bị label == 0
negative_pairs = []
for _, job in sample_jobs.iterrows():
    for _, resume in sample_resumes.iterrows():
        label = is_matching_pair(resume['category'], job['job_title'])
        if label == 0:
            negative_pairs.append({
                'resume_text': resume['clean_resume_text'],
                'job_text': job['clean_job_desc'],
                'label': 0,
                'resume_category': resume['category'],
                'job_title': job['job_title']
            })

# Shuffle và lấy số lượng negative pairs bằng với positive pairs để dataset cân bằng
random.shuffle(negative_pairs)
num_positive = len(pairs_data)
pairs_data.extend(negative_pairs[:num_positive])

# Chuyển thành DataFrame
df_golden_set = pd.DataFrame(pairs_data)

print(f"\n✅ ĐÃ TẠO THÀNH CÔNG GOLDEN TEST SET!")
print(f" Tổng số cặp: {len(df_golden_set)}")
print(f"🟢 Số cặp Match (Label 1): {len(df_golden_set[df_golden_set['label'] == 1])}")
print(f"🔴 Số cặp Mismatch (Label 0): {len(df_golden_set[df_golden_set['label'] == 0])}")

# Xem thử dữ liệu
display(df_golden_set.head(3))

BƯỚC 2: XÂY DỰNG GOLDEN TEST SET (GROUND TRUTH)
️ Đang tạo cặp dữ liệu (Có thể mất 1-2 phút)...

✅ ĐÃ TẠO THÀNH CÔNG GOLDEN TEST SET!
 Tổng số cặp: 19978
🟢 Số cặp Match (Label 1): 9989
🔴 Số cặp Mismatch (Label 0): 9989


,resume_text,job_text,label,resume_category,job_title
0,information technology specialist professional...,description mentor worldwide llc a member of j...,1,Information Technology,"Principal Engineer, RD"
1,information technology specialist i profession...,description mentor worldwide llc a member of j...,1,Information Technology,"Principal Engineer, RD"
2,information technology specialist discountpcfi...,description mentor worldwide llc a member of j...,1,Information Technology,"Principal Engineer, RD"


In [ ]:
print("=" * 60)
print("BƯỚC 3: LƯU DỮ LIỆU ")
print("=" * 60)

# Lưu Golden Test Set
df_golden_set.to_csv('../data/processed/golden_test_set.csv', index=False, encoding='utf-8')
print("💾 Đã lưu Golden Test Set tại: data/processed/golden_test_set.csv")

# Cập nhật file Jobs và Resumes với cột extracted_skills
df_jobs.to_csv('../data/processed/jobs_tech_with_skills.csv', index=False, encoding='utf-8')
df_resumes.to_csv('../data/processed/resumes_with_skills.csv', index=False, encoding='utf-8')
print("💾 Đã lưu Jobs và Resumes kèm kỹ năng.")

BƯỚC 3: LƯU DỮ LIỆU TUẦN 2
💾 Đã lưu Golden Test Set tại: data/processed/golden_test_set.csv
💾 Đã lưu Jobs và Resumes kèm kỹ năng.
